In [ ]:
# scripts/11_evaluate_model_with_distance_matrix.py

"""
Step 11 - Full evaluation of the functional k-NN model using a precomputed distance matrix.

Inputs:
- outputs/data/functional_space.parquet
- outputs/models/functional_knn_params.pkl
- outputs/matrices/functional_distance_matrix.npy
- outputs/matrices/functional_distance_matrix_nits.npy

Outputs (saved under outputs/):
- results:
  - outputs/results/step11_results.pkl
  - outputs/results/step11_roc_folds.pkl
  - outputs/results/step11_prc_folds.pkl
- figures:
  - outputs/figures/step11_mean_roc.png
  - outputs/figures/step11_mean_pr_curve.png
  - outputs/figures/step11_indicator_weights_radar.png
  - outputs/figures/step11_error_rate_by_sector_heatmap.png
  - outputs/figures/step11_error_rate_by_department_heatmap.png
- data:
  - outputs/data/step11_predictions_and_errors.parquet
- reports:
  - outputs/reports/step11_summary.tex
"""

from __future__ import annotations

import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from tqdm import tqdm
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, log_loss,
    precision_recall_curve, average_precision_score
)


# ---------------------------------------------------------------------
# Step 1: Paths
# ---------------------------------------------------------------------
SPACEF_PATH = Path("outputs") / "data" / "functional_space.parquet"
PARAMS_PATH = Path("outputs") / "models" / "functional_knn_params.pkl"
DIST_MATRIX_PATH = Path("outputs") / "matrices" / "functional_distance_matrix.npy"
NITS_ORDER_PATH = Path("outputs") / "matrices" / "functional_distance_matrix_nits.npy"

RESULTS_PATH = Path("outputs") / "results" / "step11_results.pkl"
ROC_FOLDS_PATH = Path("outputs") / "results" / "step11_roc_folds.pkl"
PRC_FOLDS_PATH = Path("outputs") / "results" / "step11_prc_folds.pkl"

FIG_MEAN_ROC_PATH = Path("outputs") / "figures" / "step11_mean_roc.png"
FIG_MEAN_PRC_PATH = Path("outputs") / "figures" / "step11_mean_pr_curve.png"
FIG_RADAR_PATH = Path("outputs") / "figures" / "step11_indicator_weights_radar.png"
PREDICTIONS_PATH = Path("outputs") / "data" / "step11_predictions_and_errors.parquet"
FIG_SECTOR_PATH = Path("outputs") / "figures" / "step11_error_rate_by_sector_heatmap.png"
FIG_DEPT_PATH = Path("outputs") / "figures" / "step11_error_rate_by_department_heatmap.png"

SUMMARY_TEX_PATH = Path("outputs") / "reports" / "step11_summary.tex"

# Ensure output folders exist
for p in [
    RESULTS_PATH, ROC_FOLDS_PATH, PRC_FOLDS_PATH,
    FIG_MEAN_ROC_PATH, FIG_MEAN_PRC_PATH, FIG_RADAR_PATH,
    PREDICTIONS_PATH, FIG_SECTOR_PATH, FIG_DEPT_PATH,
    SUMMARY_TEX_PATH
]:
    p.parent.mkdir(parents=True, exist_ok=True)


# ---------------------------------------------------------------------
# Step 2: Load data, params, and distance matrix
# ---------------------------------------------------------------------
df = pd.read_parquet(SPACEF_PATH)

with open(PARAMS_PATH, "rb") as f:
    params = pickle.load(f)

dist_matrix = np.load(DIST_MATRIX_PATH)
nits_order = np.load(NITS_ORDER_PATH, allow_pickle=True)

# Core params
k_final = int(params["k"])
lambda_final = float(params["lambda"])

# Financial weights (compatibility: weights vs pesos)
weights_final = params.get("weights", None)
if weights_final is None:
    weights_final = params.get("pesos", None)
if weights_final is None:
    raise KeyError("Params file must contain 'weights' (or legacy 'pesos').")

top3_final = params.get("top3", None)

# Extra weights (repo-friendly keys first; fallback to legacy keys)
w_dep = params.get("weight_DEP", None)
w_ciiu = params.get("weight_CIIU", None)
w_year = params.get("weight_year", None)
max_year_gap = float(params.get("max_year_gap", 24.0))

if w_dep is None:
    w_dep = params.get("peso_DEP", None)
if w_ciiu is None:
    w_ciiu = params.get("peso_CIIU", None)
if w_year is None:
    w_year = params.get("peso_desfase", None)

w_dep = float(w_dep) if w_dep is not None else 0.0
w_ciiu = float(w_ciiu) if w_ciiu is not None else 0.0
w_year = float(w_year) if w_year is not None else 0.0

# If someone stored extra weights inside the same dict (common in notebook versions),
# remove them for the "financial-only" radar plot.
EXTRA_KEYS_IN_DICT = {"DEP", "CIIU", "CIIU_Letra", "desfase", "Año_final", "year", "weight_DEP", "weight_CIIU", "weight_year"}
financial_weights = {k: float(v) for k, v in weights_final.items() if k not in EXTRA_KEYS_IN_DICT}

if len(financial_weights) == 0:
    # fallback: if we can't filter safely, assume all are financial (but this is unusual)
    financial_weights = {k: float(v) for k, v in weights_final.items()}

# Normalized financial weights for radar plot
w_sum_fin = float(sum(financial_weights.values())) if financial_weights else 1.0
weights_norm_fin = {var: (float(w) / w_sum_fin) for var, w in financial_weights.items()}
indicators_fin = sorted(weights_norm_fin.keys())

# Ensure alignment with distance matrix ordering
df = df[df["RQ_final"].notna()].copy()
if "NIT" in df.columns:
    df = df.set_index("NIT", drop=False)

df = df.loc[nits_order]
y = df["RQ_final"].values.astype(int)
NITs = df.index.tolist()

print("\n[STEP 11] Functional model evaluation using the precomputed distance matrix")
print(f"[INFO] Sample: 100% | k = {k_final} | lambda = {lambda_final:.4f}")
print(f"[INFO] Extra metric weights: w_DEP={w_dep:.4f}, w_CIIU={w_ciiu:.4f}, w_year={w_year:.4f} (max_year_gap={max_year_gap:.1f})")


def _safe_top3_names(top3_obj) -> str:
    if top3_obj is None:
        return ""
    try:
        # expected format: [(var, weight), ...]
        return ", ".join([str(v[0]) for v in top3_obj])
    except Exception:
        return ""


top3_names = _safe_top3_names(top3_final)
if top3_names:
    print(f"[INFO] Top 3 (as stored): {top3_names}")


# ---------------------------------------------------------------------
# Step 3: Cross-validation evaluation (10 folds)
# ---------------------------------------------------------------------
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

acc_list, prec_list, rec_list, f1_list, auc_list, logloss_list = [], [], [], [], [], []
roc_data, prc_data, avg_precisions = [], [], []

for fold, (train_idx, test_idx) in enumerate(tqdm(skf.split(dist_matrix, y), total=10, desc="CV folds")):
    y_test = y[test_idx]

    preds, probs = [], []
    for i in test_idx:
        distances = [(dist_matrix[i, j], y[j]) for j in train_idx]
        neighbors = sorted(distances, key=lambda x: x[0])[:k_final]
        proba = float(np.mean([v[1] for v in neighbors]))
        preds.append(int(round(proba)))
        probs.append(proba)

    acc_list.append(accuracy_score(y_test, preds))
    prec_list.append(precision_score(y_test, preds, zero_division=0))
    rec_list.append(recall_score(y_test, preds))
    f1_list.append(f1_score(y_test, preds))
    auc_list.append(roc_auc_score(y_test, probs))
    logloss_list.append(log_loss(y_test, probs))

    fpr, tpr, _ = roc_curve(y_test, probs)
    roc_data.append({"fold": fold + 1, "fpr": fpr, "tpr": tpr})

    precision, recall, _ = precision_recall_curve(y_test, probs)
    avg_precisions.append(average_precision_score(y_test, probs))
    prc_data.append({"fold": fold + 1, "precision": precision, "recall": recall})


# ---------------------------------------------------------------------
# Step 4: Mean ROC curve plot
# ---------------------------------------------------------------------
mean_fpr = np.linspace(0, 1, 100)
tprs = [np.interp(mean_fpr, r["fpr"], r["tpr"]) for r in roc_data]
mean_tpr = np.mean(tprs, axis=0)
std_tpr = np.std(tprs, axis=0)
mean_tpr[-1] = 1.0

plt.figure(figsize=(7, 5))
plt.plot(mean_fpr, mean_tpr, label="Mean ROC curve", linewidth=2)
plt.fill_between(mean_fpr, mean_tpr - std_tpr, mean_tpr + std_tpr, alpha=0.2, label="±1 SD")
plt.plot([0, 1], [0, 1], "k--", linewidth=1)
plt.xlabel("False Positive Rate (FPR)")
plt.ylabel("True Positive Rate (TPR)")
plt.title("Mean ROC curve - Functional k-NN model")
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.savefig(FIG_MEAN_ROC_PATH, dpi=200)
plt.close()

print(f"[SAVED] Mean ROC figure: {FIG_MEAN_ROC_PATH}")


# ---------------------------------------------------------------------
# Step 5: Mean Precision-Recall curve plot
# ---------------------------------------------------------------------
mean_recall = np.linspace(0, 1, 100)
precisions_interp = [np.interp(mean_recall, r["recall"][::-1], r["precision"][::-1]) for r in prc_data]
mean_precision = np.mean(precisions_interp, axis=0)
std_precision = np.std(precisions_interp, axis=0)

plt.figure(figsize=(7, 5))
plt.plot(mean_recall, mean_precision, label="Mean PR curve", linewidth=2)
plt.fill_between(mean_recall, mean_precision - std_precision, mean_precision + std_precision, alpha=0.2, label="±1 SD")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Mean Precision-Recall curve - Functional k-NN model")
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.savefig(FIG_MEAN_PRC_PATH, dpi=200)
plt.close()

print(f"[SAVED] Mean PR figure: {FIG_MEAN_PRC_PATH}")


# ---------------------------------------------------------------------
# Step 6: Summary table + results packaging
# ---------------------------------------------------------------------
f1_mean = float(np.mean(f1_list))
f1_std = float(np.std(f1_list))

performance = (
    "Very high / excellent performance" if f1_mean >= 0.95 else
    "High performance" if f1_mean >= 0.85 else
    "Moderate performance" if f1_mean >= 0.70 else
    "Low performance" if f1_mean >= 0.50 else
    "Very low performance"
)

robustness = (
    "Highly robust and stable" if f1_std <= 0.0003 else
    "Very stable" if f1_std <= 0.002 else
    "Good stability" if f1_std <= 0.007 else
    "Moderate variability" if f1_std <= 0.015 else
    "High variability / less robust"
)

observation = f"{performance}. {robustness}."

# Prepare Top3 lines but only if they are in financial weights
top3_lines = None
if top3_final is not None:
    try:
        top3_lines = []
        for var, raw_w in top3_final:
            var = str(var)
            if var in weights_norm_fin:
                top3_lines.append(
                    f"{var} (raw weight: {float(raw_w):.3f}, normalized: {weights_norm_fin[var]*100:.1f}%)"
                )
        if len(top3_lines) == 0:
            top3_lines = None
    except Exception:
        top3_lines = None

summary_final = {
    "Accuracy": f"{np.mean(acc_list):.4f} ±{np.std(acc_list):.4f}",
    "Precision": f"{np.mean(prec_list):.4f} ±{np.std(prec_list):.4f}",
    "Recall": f"{np.mean(rec_list):.4f} ±{np.std(rec_list):.4f}",
    "F1-score": f"{f1_mean:.4f} ±{f1_std:.4f}",
    "AUC": f"{np.mean(auc_list):.4f} ±{np.std(auc_list):.4f}",
    "Log Loss": f"{np.mean(logloss_list):.4f} ±{np.std(logloss_list):.4f}",
    "k": k_final,
    "lambda": lambda_final,
    "w_DEP": w_dep,
    "w_CIIU": w_ciiu,
    "w_year": w_year,
    "max_year_gap": max_year_gap,
    "Observation": observation,
}

print("\n[SUMMARY]")
for k_, v_ in summary_final.items():
    print(f" - {k_}: {v_}")

# Save fold-level objects
with open(RESULTS_PATH, "wb") as f:
    pickle.dump(
        {
            "summary": summary_final,
            "folds_accuracy": acc_list,
            "folds_precision": prec_list,
            "folds_recall": rec_list,
            "folds_f1": f1_list,
            "folds_auc": auc_list,
            "folds_logloss": logloss_list,
            "folds_avg_precision": avg_precisions,
            "params": {
                "k": k_final,
                "lambda": lambda_final,
                "weights_financial": financial_weights,
                "weights_raw": weights_final,
                "top3": top3_final,
                "w_DEP": w_dep,
                "w_CIIU": w_ciiu,
                "w_year": w_year,
                "max_year_gap": max_year_gap,
            },
            "nits": NITs,
        },
        f,
    )

with open(ROC_FOLDS_PATH, "wb") as f:
    pickle.dump(roc_data, f)

with open(PRC_FOLDS_PATH, "wb") as f:
    pickle.dump(prc_data, f)

print(f"[SAVED] Results pickle: {RESULTS_PATH}")
print(f"[SAVED] ROC folds pickle: {ROC_FOLDS_PATH}")
print(f"[SAVED] PR folds pickle: {PRC_FOLDS_PATH}")


# ---------------------------------------------------------------------
# Step 7: Radar plot - normalized importance (financial indicators only)
# ---------------------------------------------------------------------
variables = list(weights_norm_fin.keys())
values = [weights_norm_fin[v] for v in variables]
if len(variables) >= 3:
    values_closed = values + [values[0]]
    angles = np.linspace(0, 2 * np.pi, len(variables), endpoint=False).tolist()
    angles += angles[:1]

    fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
    ax.plot(angles, values_closed, linewidth=2)
    ax.fill(angles, values_closed, alpha=0.2)
    ax.set_thetagrids(np.degrees(angles[:-1]), variables, fontsize=9)
    ax.set_title("Relative importance of financial indicators (normalized weights)", y=1.10)
    plt.tight_layout()
    plt.savefig(FIG_RADAR_PATH, dpi=200)
    plt.close()
    print(f"[SAVED] Radar plot: {FIG_RADAR_PATH}")
else:
    print("[WARN] Radar plot skipped: not enough financial indicators to plot.")


# ---------------------------------------------------------------------
# Step 8: Full-sample predictions (leave-one-out over the distance matrix)
# ---------------------------------------------------------------------
df_pred = pd.read_parquet(SPACEF_PATH)
df_pred = df_pred[df_pred["RQ_final"].notna()].copy()
if "NIT" in df_pred.columns:
    df_pred = df_pred.set_index("NIT", drop=False)

df_pred = df_pred.loc[nits_order]
y_real = df_pred["RQ_final"].values.astype(int)

preds = []
for i in tqdm(range(len(y_real)), desc="Predicting firms (leave-one-out via distance matrix)"):
    distances = [(dist_matrix[i, j], y_real[j]) for j in range(len(y_real)) if j != i]
    neighbors = sorted(distances, key=lambda x: x[0])[:k_final]
    proba = float(np.mean([v[1] for v in neighbors]))
    preds.append(int(round(proba)))

df_pred["pred_RQ"] = preds
df_pred["error"] = df_pred["pred_RQ"] != df_pred["RQ_final"]

df_pred.reset_index(drop=True).to_parquet(PREDICTIONS_PATH, index=False)
print(f"[SAVED] Predictions and errors: {PREDICTIONS_PATH}")


# ---------------------------------------------------------------------
# Step 9: Heatmaps - error rate by sector and department
# ---------------------------------------------------------------------
sector_map = {
    "A": "Agriculture", "B": "Mining", "C": "Manufacturing", "D": "Electricity",
    "E": "Water & waste", "F": "Construction", "G": "Trade & vehicles",
    "H": "Transportation", "I": "Accommodation & food", "J": "Information & communication",
    "K": "Finance & insurance", "L": "Real estate", "M": "Professional & technical",
    "N": "Administrative services", "O": "Government", "P": "Education",
    "Q": "Health & social work", "R": "Arts & recreation", "S": "Other services",
    "T": "Households as employers", "U": "Extraterritorial organizations",
    np.nan: "Unclassified",
}

# Sector heatmap
if "CIIU_Letra" in df_pred.columns:
    ciiu_counts = df_pred["CIIU_Letra"].value_counts()
    valid_letters = ciiu_counts[ciiu_counts >= 50].index.tolist()

    df_pred["Sector"] = df_pred["CIIU_Letra"].map(sector_map)
    sector_errors = (
        df_pred[df_pred["CIIU_Letra"].isin(valid_letters)]
        .groupby("Sector")["error"]
        .mean()
        .sort_values(ascending=False)
    )

    plt.figure(figsize=(12, 5))
    sns.heatmap(sector_errors.to_frame().T, cmap="Purples", annot=True, fmt=".2f", cbar=False)
    plt.title("Error rate by economic sector")
    plt.tight_layout()
    plt.savefig(FIG_SECTOR_PATH, dpi=200)
    plt.close()
    print(f"[SAVED] Sector heatmap: {FIG_SECTOR_PATH}")

# Department heatmap
if "DEP" in df_pred.columns:
    dep_counts = df_pred["DEP"].value_counts()
    valid_deps = dep_counts[dep_counts >= 20].index.tolist()

    dep_errors = (
        df_pred[df_pred["DEP"].isin(valid_deps)]
        .groupby("DEP")["error"]
        .mean()
        .sort_values(ascending=False)
    )

    plt.figure(figsize=(14, 5))
    sns.heatmap(dep_errors.to_frame().T, cmap="Reds", annot=True, fmt=".2f", cbar=False)
    plt.title("Average error rate by department")
    plt.tight_layout()
    plt.savefig(FIG_DEPT_PATH, dpi=200)
    plt.close()
    print(f"[SAVED] Department heatmap: {FIG_DEPT_PATH}")


# ---------------------------------------------------------------------
# Step 10: Save LaTeX summary (English)
# ---------------------------------------------------------------------
with open(SUMMARY_TEX_PATH, "w", encoding="utf-8") as f:
    f.write(r"\section*{Step 11: Functional Model Evaluation Using the Distance Matrix}" + "\n")
    f.write(r"\begin{itemize}" + "\n")
    f.write(r"  \item The functional k-NN model was evaluated using 10-fold stratified cross-validation.\n")
    f.write(f"  \item Mean F1-score: {f1_mean:.4f} $\\pm$ {f1_std:.4f}\n")
    f.write(f"  \item Observation: {observation}\n")
    f.write(r"  \item The evaluation leveraged a precomputed distance matrix computed with the extended metric.\n")
    f.write(f"  \item Extra metric weights: DEP={w_dep:.4f}, CIIU={w_ciiu:.4f}, year={w_year:.4f} (max year gap={max_year_gap:.1f}).\n")
    if top3_lines is not None:
        f.write(r"  \item Top 3 financial indicators by weight:\n")
        f.write(r"  \begin{itemize}" + "\n")
        for line in top3_lines:
            f.write(f"    \\item {line}\n")
        f.write(r"  \end{itemize}" + "\n")
    f.write(r"  \item Generated outputs:\n")
    f.write(r"  \begin{itemize}" + "\n")
    f.write(r"    \item \texttt{outputs/results/step11\_results.pkl}\n")
    f.write(r"    \item \texttt{outputs/results/step11\_roc\_folds.pkl}\n")
    f.write(r"    \item \texttt{outputs/results/step11\_prc\_folds.pkl}\n")
    f.write(r"    \item \texttt{outputs/figures/step11\_mean\_roc.png}\n")
    f.write(r"    \item \texttt{outputs/figures/step11\_mean\_pr\_curve.png}\n")
    f.write(r"    \item \texttt{outputs/figures/step11\_indicator\_weights\_radar.png}\n")
    f.write(r"    \item \texttt{outputs/data/step11\_predictions\_and\_errors.parquet}\n")
    f.write(r"    \item \texttt{outputs/figures/step11\_error\_rate\_by\_sector\_heatmap.png}\n")
    f.write(r"    \item \texttt{outputs/figures/step11\_error\_rate\_by\_department\_heatmap.png}\n")
    f.write(r"  \end{itemize}" + "\n")
    f.write(r"\end{itemize}" + "\n")

print(f"[SAVED] LaTeX summary: {SUMMARY_TEX_PATH}")